# TradingBot — Signal & Trade Analysis

Run from the project root (`trading_bot/`), not from inside `notebooks/`.

In [ ]:
import sys, os
# Add project root to path so bot.* imports work
sys.path.insert(0, os.path.abspath('..'))

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from bot.config.settings import settings

# Connect via standard sqlite3 (read-only)
conn = sqlite3.connect(settings.DATABASE_PATH)
conn.row_factory = sqlite3.Row

print(f"Connected to: {settings.DATABASE_PATH}")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables:", tables['name'].tolist())

In [ ]:
# ── Cell 2: Load signals ───────────────────────────────────────────────────
signals = pd.read_sql(
    "SELECT * FROM signals ORDER BY timestamp ASC", conn
)
# SQLite stores timestamp as Unix ms integer
signals['datetime'] = pd.to_datetime(signals['timestamp'], unit='ms', utc=True)
signals['date'] = signals['datetime'].dt.date

print(f"Total signals: {len(signals)}")
print(signals['direction'].value_counts().to_string())
signals.tail(5)

In [ ]:
# ── Cell 3: Signal confidence over time ───────────────────────────────────
actionable = signals[signals['direction'] != 'HOLD'].copy()

if actionable.empty:
    print("No BUY/SELL signals yet — run the bot for a while first.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    for direction, grp in actionable.groupby('direction'):
        color = 'green' if direction == 'BUY' else 'red'
        axes[0].scatter(grp['datetime'], grp['confidence'],
                        color=color, alpha=0.7, label=direction, s=25)

    axes[0].axhline(y=settings.SIGNAL_THRESHOLD, color='orange',
                    linestyle='--', label=f'Threshold ({settings.SIGNAL_THRESHOLD:.0%})')
    axes[0].set_ylabel('Confidence')
    axes[0].set_title('Signal Confidence Over Time')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    daily = actionable.groupby(['date', 'direction']).size().unstack(fill_value=0)
    daily.plot(kind='bar', ax=axes[1],
               color={'BUY': 'green', 'SELL': 'red'}, alpha=0.7)
    axes[1].set_ylabel('Count')
    axes[1].set_title('Daily Signal Counts')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('signal_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to notebooks/signal_analysis.png")

In [ ]:
# ── Cell 4: Trade history & realized PnL ──────────────────────────────────
trades = pd.read_sql(
    "SELECT * FROM trades ORDER BY opened_at ASC", conn
)
closed = trades[trades['status'] == 'closed'].copy()
open_t = trades[trades['status'] == 'open'].copy()

print(f"Open trades:   {len(open_t)}")
print(f"Closed trades: {len(closed)}")

if not closed.empty:
    wins   = closed[closed['pnl_usdt'] > 0]
    losses = closed[closed['pnl_usdt'] <= 0]
    print(f"Wins: {len(wins)}  Losses: {len(losses)}  "
          f"Win rate: {len(wins)/len(closed):.0%}")
    print(f"Total PnL: {closed['pnl_usdt'].sum():+.2f} USDT")

    # Cumulative PnL curve
    closed['closed_dt'] = pd.to_datetime(closed['closed_at'])
    closed_sorted = closed.sort_values('closed_dt')
    closed_sorted['cum_pnl'] = closed_sorted['pnl_usdt'].cumsum()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(closed_sorted['closed_dt'], closed_sorted['cum_pnl'],
            color='teal', linewidth=2)
    ax.fill_between(closed_sorted['closed_dt'], closed_sorted['cum_pnl'],
                    alpha=0.1, color='teal')
    ax.axhline(0, color='gray', linestyle='--')
    ax.set_ylabel('Cumulative PnL (USDT)')
    ax.set_title('Realized PnL Curve')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No closed trades yet.")

In [ ]:
# ── Cell 5: XGBoost feature importances ───────────────────────────────────
from bot.model.trainer import load_model, FEATURE_COLS

try:
    model = load_model()
    feat_imp = pd.Series(
        model.feature_importances_, index=FEATURE_COLS
    ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_xlabel('Importance (gain)')
    ax.set_title('XGBoost Feature Importances')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    print(feat_imp.sort_values(ascending=False).to_string())
except FileNotFoundError as e:
    print(f"Model not found: {e}\nRun: python -m bot.model.trainer")

In [ ]:
# ── Cell 6: Retrain model with fresh data ─────────────────────────────────
# Review the walk-forward metrics before restarting the live bot.

from bot.data.storage import get_candles
from bot.features.indicators import compute_indicators
from bot.model.labels import generate_labels
from bot.model.trainer import prepare_features, train_model, save_model

print(f"Fetching candles: {settings.SYMBOL}/{settings.TIMEFRAME}")
raw = get_candles(settings.SYMBOL, settings.TIMEFRAME, limit=5000)
print(f"  {len(raw)} candles loaded from DB")

if len(raw) < 200:
    print("Not enough data. Run the bot for longer before retraining.")
else:
    df = compute_indicators(raw)
    df = generate_labels(df, forward_candles=4, threshold=0.003)
    X, y = prepare_features(df)
    print(f"  Feature matrix: {X.shape}")

    model = train_model(X, y)
    save_model(model)
    print(f"\nModel saved → {settings.MODEL_PATH}")
    print("Restart python -m bot.loop to use new weights.")